# Trajectory Drawing Tool (interactive notebook for VSCode)

**How to use:**
1. The parameters in Cell 1 (`INPUT_IMG_PATH`, `SAVE_DIR`) are loaded automatically when launched via `inference.sh`.
   You may also set them manually if you want to use this notebook standalone.
2. Run all cells in order.
3. On the satellite image, **hold the left mouse button and drag** to draw a trajectory; release to finish.
4. Run the last cell to save `trajectory.csv`.

**Prerequisite:** `pip install ipympl` (only needed the first time).


In [ ]:
# === Cell 1: Environment setup + load parameters ===
%matplotlib widget
import os, json
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import splprep, splev

# inference.sh writes the trajectory config to one of the following paths (absolute paths inside).
CONFIG_CANDIDATES = [
    './results/demo/.traj_config.json',
    '../results/demo/.traj_config.json',
    '../../results/demo/.traj_config.json',
]

INPUT_IMG_PATH = None
SAVE_DIR       = None
NUM_OF_POINT   = 79

cfg_path = None
for c in CONFIG_CANDIDATES:
    if os.path.exists(c):
        cfg_path = os.path.abspath(c)
        break

if cfg_path is not None:
    with open(cfg_path, 'r') as f:
        cfg = json.load(f)
    INPUT_IMG_PATH = cfg.get('input_img_path')
    SAVE_DIR       = cfg.get('save_dir') or cfg.get('work_dir')
    NUM_OF_POINT   = int(cfg.get('num_points', NUM_OF_POINT))
    print(f'[OK] Loaded config from: {cfg_path}')
else:
    # Fallback: environment variables (only available when jupyter is launched from the same shell)
    INPUT_IMG_PATH = os.environ.get('TRAJ_INPUT_IMG')
    SAVE_DIR       = os.environ.get('TRAJ_SAVE_DIR') or os.environ.get('TRAJ_WORK_DIR')
    NUM_OF_POINT   = int(os.environ.get('TRAJ_NUM_POINTS', str(NUM_OF_POINT)))
    print('[WARN] No config file found, falling back to environment variables.')

if INPUT_IMG_PATH is None or not os.path.exists(INPUT_IMG_PATH):
    print('[ERROR] Could not auto-resolve input image path.')
    print('        Please set INPUT_IMG_PATH and SAVE_DIR manually below,')
    print('        or run `bash inference.sh <your_image>` first and then re-open this notebook.')
    # INPUT_IMG_PATH = '/abs/path/to/sat.png'
    # SAVE_DIR       = '/abs/path/to/results/demo/<sat_stem>'
else:
    if SAVE_DIR is None:
        SAVE_DIR = os.path.join('./results/demo', os.path.basename(INPUT_IMG_PATH).rsplit('.', 1)[0])
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f'Input image : {INPUT_IMG_PATH}')
    print(f'Save dir    : {SAVE_DIR}')
    print(f'Num points  : {NUM_OF_POINT}')


In [ ]:
# === Cell 2: Show satellite image and draw trajectory by left-click drag ===
sat_image = plt.imread(INPUT_IMG_PATH)

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(sat_image)
ax.set_title('Hold left mouse button and drag to draw a trajectory; release when done')
ax.set_axis_off()

coords = []
state = {'dragging': False, 'done': False, 'line': None}

def on_press(event):
    if event.button == 1 and event.inaxes == ax:
        state['dragging'] = True
        coords.clear()
        coords.append((event.xdata, event.ydata))
        if state['line'] is not None:
            state['line'].remove()
        state['line'], = ax.plot([event.xdata], [event.ydata], '-', color='red', linewidth=2)
        fig.canvas.draw_idle()

def on_drag(event):
    if not state['dragging'] or event.button != 1 or event.xdata is None or event.ydata is None:
        return
    coords.append((event.xdata, event.ydata))
    xs, ys = zip(*coords)
    state['line'].set_data(xs, ys)
    fig.canvas.draw_idle()

def on_release(event):
    if event.button == 1 and state['dragging']:
        state['dragging'] = False
        state['done'] = True
        ax.set_title(f'Captured {len(coords)} points. Run the next cell to fit and preview.')
        fig.canvas.draw_idle()
        print(f'[OK] Captured {len(coords)} points.')

fig.canvas.mpl_connect('button_press_event', on_press)
fig.canvas.mpl_connect('motion_notify_event', on_drag)
fig.canvas.mpl_connect('button_release_event', on_release)
plt.show()


In [ ]:
# === Cell 3: Fit a smooth spline through the captured points and preview ===
assert len(coords) >= 4, f'Too few points captured ({len(coords)}); need at least 4. Re-draw in the previous cell.'

pixels = np.array(list(dict.fromkeys(coords)))  # de-duplicate while preserving order
tck, u = splprep(pixels.T, s=25, per=0)
u_new = np.linspace(u.min(), u.max(), NUM_OF_POINT + 1)
x_new, y_new = splev(u_new, tck)
smooth_path = np.array([x_new, y_new]).T
angles = np.arctan2(y_new[1:] - y_new[:-1], x_new[1:] - x_new[:-1])

# Preview
fig_final, ax_final = plt.subplots(figsize=(8, 8))
ax_final.imshow(sat_image)
ax_final.plot(pixels[:, 0], pixels[:, 1], 'o', color='red', markersize=3, label='captured points')
ax_final.plot(smooth_path[:, 0], smooth_path[:, 1], '-', color='blue', linewidth=2, label='smoothed path')
ax_final.legend()
ax_final.set_axis_off()
plt.show()

print(f'Captured points: {len(pixels)}, smoothed points: {len(smooth_path)}')


In [ ]:
# === Cell 4: Save trajectory.csv ===
os.makedirs(SAVE_DIR, exist_ok=True)

save_csv = os.path.join(SAVE_DIR, 'trajectory.csv')
fig_final.savefig(os.path.join(SAVE_DIR, 'trajectory.png'), dpi=300, bbox_inches='tight')

with open(save_csv, 'w') as f:
    f.write('w,h,angle\n')
    for i in range(len(smooth_path) - 1):
        f.write(f'{smooth_path[i][0]},{smooth_path[i][1]},{angles[i]}\n')

print(f'[OK] Trajectory saved to: {os.path.abspath(save_csv)}')
print(f'[OK] Visualization saved: {os.path.abspath(os.path.join(SAVE_DIR, "trajectory.png"))}')
print('\nYou can now return to the terminal; inference.sh will detect this file and continue.')
